# 01b — Dataset Integrity Audit & Cohort Lock (Phase 1 gate) — v3 (Kaggle confirmation run)
**Project:** From Prediction to Referral — Uncertainty-Aware Pneumonia Detection under Cross-Population Distribution Shift

Audits Notebook 01 output (`mooney_manifest_final.csv`) against the raw Kaggle data and **locks the NIH OOD cohorts
from metadata rules alone, before any model exists**. Trains nothing. Reads no model output.

**Locked protocol**
* Mooney development pool = original `train/` + original 16-image `val/` → group-aware train/val split on heuristic patient IDs.
* Mooney original `test/` = untouched ID test (`final_split == "test_id"`).
* NIH primary cohorts: 18 ≤ age ≤ 100 (`OOD_AP_ADULT`, `OOD_PA_ADULT`, `OOD_HARD_AP_ADULT`).
* NIH sensitivity cohorts: no age restriction (`OOD_AP_ALLAGE`, `OOD_PA_ALLAGE`, `OOD_HARD_AP_ALLAGE`).
* NIH is external-only; patients in both classes are kept; all NIH CIs later use patient-level bootstrap.

**Duplicate policy:** SHA-256 identity across splits → FAIL · same image with conflicting labels → FAIL ·
dHash candidates → WARN (inspect, never auto-delete) · raw `chest_xray/chest_xray/` or `__MACOSX` trees → FAIL only if the manifest references them.

Final output: exactly one verdict — `PASS — Notebook 02 may proceed` or `FAIL — do NOT train until fixed`.

In [ ]:
import os, re, json, hashlib, sys, platform, datetime, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

# ------------------------------------------------------------------ CONFIG (edit paths only)
MOONEY_ROOT     = Path("/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray")  # contains train/ val/ test/
NIH_ROOT        = Path("/kaggle/input/datasets/nih-chest-xrays/data")      # Data_Entry_2017.csv, images_0xx/images/
PROJECT         = Path("/kaggle/working/CXR_OOD_Project")
N01_DIR         = Path("/kaggle/input/cxr-ood-notebook01-output")          # EDIT: folder holding the Notebook 01 output CSVs
MOONEY_MANIFEST = N01_DIR / "mooney_manifest_final.csv"
NIH_N01_MANIFEST_DIR = N01_DIR
# Fingerprints of the cohorts locked by the offline Phase 1 audit (2026-10-05). The Kaggle run must reproduce them.
EXPECTED_FINGERPRINTS = {
    "OOD_AP_ALLAGE": "2f209001f1183bcddc265cd93a36eaf00d2151d6d0088b0b44c2cd1d03c70f80",
    "OOD_AP_ADULT": "b13bc8c6413b3a5725e08184ade4a34305288d0daff717533589399c83e0d567",
    "OOD_PA_ALLAGE": "f1a6c96829c1b91d1221cd971c2de053be878f49fea846ff5fd36ba3e36b8474",
    "OOD_PA_ADULT": "ede8eeb671334525f5d4d0edce8a8a025bcf7992bb19d9060e8a17c6bd116cb0",
    "OOD_HARD_AP_ALLAGE": "9647dd49e598ef43d1321fe31e4401786c8293e15ba71956d8b6c77f2c922372",
    "OOD_HARD_AP_ADULT": "b8cc02779f56de273cd8ffcca8bc5915451e279f97f9b42f2211c1f1c544fb60"
}

OUT_DIR     = PROJECT / "logs" / "audit"
COHORT_DIR  = PROJECT / "manifests" / "locked_cohorts"
ZIP_BASE    = PROJECT / "phase1_audit_outputs"
for d in (OUT_DIR, COHORT_DIR): d.mkdir(parents=True, exist_ok=True)

HARD_NEG_FINDINGS = {"Consolidation", "Infiltration", "Edema", "Atelectasis", "Effusion"}
AGE_MIN, AGE_MAX  = 18, 100
HASH_NIH_IMAGES   = True     # SHA-256 of every locked NIH image vs Mooney (slow: ~10–20 min on Kaggle)
RUN_NEAR_DUP      = True
DHASH_MAX_HAMMING = 4
IMG_EXT = {".jpeg", ".jpg", ".png"}
SPLIT_MAP = {"train": "train", "val": "val", "test": "test_id"}

RESULTS = []
def record(check, status, detail, blocking=True, data=None):
    assert status in {"PASS", "FAIL", "WARN", "INFO"}
    RESULTS.append(dict(check=check, status=status, blocking=bool(blocking and status == "FAIL"), detail=detail, data=data))
    print(f"[{status:4}] {check}: {detail}")

## Helpers

In [ ]:
PATH_COLS  = ["filepath", "file_path", "path", "image_path", "img_path", "full_path"]
FNAME_COLS = ["filename", "file_name", "image_index", "Image Index", "image", "fname"]
PID_COLS   = ["patient_id", "Patient ID", "patient", "pid"]

def find_col(df, cands, required=True):
    lower = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in lower: return lower[c.lower()]
    if required: raise KeyError(f"None of {cands} in columns {list(df.columns)}")
    return None

LABEL_MAP = {"PNEUMONIA": 1, "NORMAL": 0, "NO_FINDING": 0, "NO FINDING": 0, "NON_PNEUMONIA_ABNORMAL": 0}

def labels_of(df, where):
    """Numeric `y` wins whenever it is present and valid; string label is only a fallback / cross-check."""
    if "y" in df.columns:
        y = pd.to_numeric(df["y"], errors="coerce")
        bad = ~y.isin([0, 1])
        if bad.any():
            record(f"{where}:y_valid", "FAIL", f"{int(bad.sum())} rows with y not in {{0,1}}")
        else:
            record(f"{where}:y_valid", "PASS", "numeric y used as label source")
        if "label" in df.columns:
            s = df["label"].astype(str).str.strip().str.upper().map(LABEL_MAP)
            dis = (s.notna() & ~bad & (s != y)).sum()
            record(f"{where}:y_vs_label_string", "PASS" if dis == 0 else "FAIL",
                   f"{int(dis)} rows where string label disagrees with y (y retained as source of truth)")
        return y.fillna(-1).astype(int)
    col = find_col(df, ["label", "target", "class"])
    s = df[col].astype(str).str.strip().str.upper().map(LABEL_MAP)
    record(f"{where}:label_parse", "PASS" if s.notna().all() else "FAIL",
           f"no y column; parsed '{col}', {int(s.isna().sum())} unrecognised values")
    return s.fillna(-1).astype(int)

def sha256(path, buf=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(buf), b""): h.update(chunk)
    return h.hexdigest()

def mooney_pid(fname):
    f = fname.upper()
    for pat, pre in [(r"^(PERSON\d+)_", "P:"), (r"^(NORMAL2-IM-\d+)-", "N:"), (r"^(IM-\d+)-", "N:")]:
        m = re.match(pat, f)
        if m: return pre + m.group(1)
    return None

def originals(sub):
    return {p.resolve() for p in (MOONEY_ROOT / sub).rglob("*") if p.suffix.lower() in IMG_EXT}

def finalize():
    res = pd.DataFrame(RESULTS)
    blocking = res[res.blocking]
    verdict = "PASS — Notebook 02 may proceed" if blocking.empty else "FAIL — do NOT train until fixed"
    env = dict(python=sys.version, platform=platform.platform(), pandas=pd.__version__, numpy=np.__version__,
               timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
    cfg = dict(MOONEY_ROOT=str(MOONEY_ROOT), NIH_ROOT=str(NIH_ROOT), MOONEY_MANIFEST=str(MOONEY_MANIFEST),
               AGE_RANGE=[AGE_MIN, AGE_MAX], HARD_NEG_FINDINGS=sorted(HARD_NEG_FINDINGS),
               DHASH_MAX_HAMMING=DHASH_MAX_HAMMING, HASH_NIH_IMAGES=HASH_NIH_IMAGES)
    (OUT_DIR / "audit_report.json").write_text(json.dumps(
        dict(verdict=verdict, n_blocking=len(blocking), blocking_issues=blocking[["check", "detail"]].to_dict("records"),
             cohort_fingerprints=globals().get("FINGERPRINTS", {}), env=env, config=cfg, results=RESULTS),
        indent=2, default=str))
    res.drop(columns="data").to_csv(OUT_DIR / "audit_checks.csv", index=False)
    if COHORT_DIR.exists():
        shutil.copytree(COHORT_DIR, OUT_DIR / "locked_cohorts", dirs_exist_ok=True)
    shutil.make_archive(str(ZIP_BASE), "zip", OUT_DIR)
    print("\n" + "=" * 72 + f"\n{verdict}\n" + "=" * 72)
    print(res.status.value_counts().to_string())
    if not blocking.empty:
        print("\nBlocking issues:")
        for _, r in blocking.iterrows(): print(f"  - {r['check']}: {r['detail']}")
    print(f"\nZIP → {ZIP_BASE}.zip")
    return verdict

## 1. Mooney — load single manifest and split by `final_split`

In [ ]:
MOONEY_OK = MOONEY_MANIFEST.exists()
record("mooney:manifest_exists", "PASS" if MOONEY_OK else "FAIL", str(MOONEY_MANIFEST))
if MOONEY_OK:
    dfm = pd.read_csv(MOONEY_MANIFEST)
    if "final_split" not in dfm.columns:
        record("mooney:final_split_column", "FAIL", f"columns = {list(dfm.columns)}"); MOONEY_OK = False
    else:
        vals = set(dfm["final_split"].astype(str))
        unexpected = vals - set(SPLIT_MAP.values())
        record("mooney:final_split_values", "PASS" if not unexpected else "FAIL",
               f"values={sorted(vals)}; unexpected={sorted(unexpected)}")
if not MOONEY_OK:
    finalize(); raise SystemExit("Mooney manifest unusable — see audit_checks.csv")

pc = find_col(dfm, PATH_COLS)
dfm["_path"] = dfm[pc].astype(str)
dfm["_abs"]  = dfm["_path"].apply(lambda p: Path(p) if Path(p).is_absolute() else MOONEY_ROOT / p)
dfm["_fname"] = dfm["_abs"].apply(lambda p: p.name)
dfm["_y"] = labels_of(dfm, "mooney")

mo = {k: dfm[dfm["final_split"].astype(str) == v].copy() for k, v in SPLIT_MAP.items()}
for k, df in mo.items():
    record(f"mooney:{k}:non_empty", "PASS" if len(df) else "FAIL",
           f"final_split='{SPLIT_MAP[k]}': n={len(df)}, PNEUMONIA={int((df._y==1).sum())}, NORMAL={int((df._y==0).sum())}")
    miss = int((~df["_abs"].apply(Path.exists)).sum())
    record(f"mooney:{k}:files_exist", "PASS" if miss == 0 else "FAIL", f"{miss} missing of {len(df)}")
    dup = int(df["_abs"].astype(str).duplicated().sum())
    record(f"mooney:{k}:no_duplicate_rows", "PASS" if dup == 0 else "FAIL", f"{dup} duplicated paths")
    fy = df["_abs"].apply(lambda p: {"PNEUMONIA": 1, "NORMAL": 0}.get(p.parent.name.upper(), -1))
    bad = int((fy != df["_y"]).sum())
    record(f"mooney:{k}:label_matches_folder", "PASS" if bad == 0 else "FAIL", f"{bad} rows disagree with class folder")
cross = int(dfm["_abs"].astype(str).duplicated().sum())
record("mooney:path_in_one_split_only", "PASS" if cross == 0 else "FAIL", f"{cross} paths listed more than once overall")

### 1a. Duplicate-tree references (raw tree existing is fine; referencing it is not)

In [ ]:
raw_nested = (MOONEY_ROOT / "chest_xray").exists() or any(MOONEY_ROOT.parent.glob("**/__MACOSX"))
record("mooney:raw_duplicate_tree_present", "INFO",
       f"nested chest_xray/chest_xray or __MACOSX present in raw input: {raw_nested} (not a failure by itself)")
ref = dfm["_abs"].astype(str).str.contains(r"chest_xray/chest_xray/|__MACOSX", regex=True)
record("mooney:manifest_references_duplicate_tree", "PASS" if not ref.any() else "FAIL",
       f"{int(ref.sum())} manifest rows reference the duplicate tree", data=dfm.loc[ref, "_path"].head(20).tolist())

### 1b. Locked protocol: dev pool = original train + original val; test_id = original test, untouched

In [ ]:
O = {s: originals(s) for s in ["train", "val", "test"]}
dev_orig = O["train"] | O["val"]
dev_man  = {p.resolve() for k in ["train", "val"] for p in mo[k]["_abs"]}
test_man = {p.resolve() for p in mo["test"]["_abs"]}

foreign = dev_man - dev_orig
record("mooney:dev_pool_only_from_orig_train_val", "PASS" if not foreign else "FAIL",
       f"{len(foreign)} train/val files not from original train/ or val/", data=[str(p) for p in list(foreign)[:20]])
dropped = dev_orig - dev_man
record("mooney:dev_pool_complete", "PASS" if not dropped else "WARN",
       f"original train+val={len(dev_orig)}, used={len(dev_man)}, dropped={len(dropped)} "
       f"(drops must be documented in Notebook 01)", blocking=False, data=[str(p) for p in list(dropped)[:20]])
extra, lost = test_man - O["test"], O["test"] - test_man
record("mooney:test_id_equals_original_test", "PASS" if not extra and not lost else "FAIL",
       f"original test/={len(O['test'])}, test_id={len(test_man)}, added={len(extra)}, dropped={len(lost)}")
leak = dev_man & O["test"]
record("mooney:original_test_not_in_dev", "PASS" if not leak else "FAIL", f"{len(leak)} original test files in train/val")

val16 = pd.DataFrame({"file": sorted(str(p) for p in O["val"])})
lookup = {str(p.resolve()): k for k in mo for p in mo[k]["_abs"]}
val16["final_split"] = val16["file"].map(lookup).fillna("NOT_IN_MANIFEST")
val16["patient_id"] = val16["file"].apply(lambda f: mooney_pid(Path(f).name))
val16.to_csv(OUT_DIR / "original_val16_disposition.csv", index=False)
record("mooney:original_val_disposition", "INFO" if (val16.final_split != "NOT_IN_MANIFEST").all() else "WARN",
       f"{len(val16)} original val images → {val16.final_split.value_counts().to_dict()}", blocking=False)

### 1c. Heuristic patient IDs (filename-derived — limitation preserved in manuscript)

In [ ]:
for k, df in mo.items():
    df["_pid"] = df["_fname"].apply(mooney_pid)
    un = int(df["_pid"].isna().sum())
    record(f"mooney:{k}:pid_resolved", "PASS" if un == 0 else "WARN", f"{un}/{len(df)} filenames without derivable ID",
           blocking=False, data=df.loc[df._pid.isna(), "_fname"].head(20).tolist())
    c = find_col(df, PID_COLS, required=False)
    if c:
        norm = lambda s: s.fillna("").astype(str).str.upper().str.replace(r"^[PN]:", "", regex=True)
        mm = int((norm(df[c]) != norm(df["_pid"])).sum())
        record(f"mooney:{k}:pid_matches_notebook01", "PASS" if mm == 0 else "WARN",
               f"{mm} rows where Notebook 01 patient ID differs from audit parser", blocking=False)
P = {k: set(mo[k]["_pid"].dropna()) for k in mo}
tv = P["train"] & P["val"]
record("mooney:patient_overlap_train_val", "PASS" if not tv else "FAIL", f"{len(tv)} shared heuristic IDs", data=sorted(tv)[:30])
for a in ["train", "val"]:
    ov = P[a] & P["test"]
    record(f"mooney:patient_overlap_{a}_test", "PASS" if not ov else "WARN",
           f"{len(ov)} shared heuristic IDs — limitation only; test is not modified", blocking=False, data=sorted(ov)[:30])
for k, df in mo.items():
    df["_sub"] = df["_fname"].str.extract(r"_(bacteria|virus)_", flags=re.I)[0].str.lower().fillna("normal")
    df["_pid_sub"] = df["_pid"] + "|" + df["_sub"]
PSUB = {k: set(mo[k]["_pid_sub"].dropna()) for k in mo}
for a in ["train", "val"]:
    ov = PSUB[a] & PSUB["test"]
    record(f"mooney:patient_overlap_{a}_test_subtype_aware", "PASS" if not ov else "WARN",
           f"{len(ov)} shared IDs using personN+bacteria/virus key (personN numbering appears to restart per subtype)",
           blocking=False, data=sorted(ov)[:30])

### 1d. SHA-256 identity

In [ ]:
for k, df in mo.items():
    df["_sha"] = [sha256(p) for p in df["_abs"]]
    w = int(df["_sha"].duplicated().sum())
    record(f"mooney:{k}:within_split_identical", "PASS" if w == 0 else "WARN", f"{w} identical images inside split", blocking=False)
allm = pd.concat(mo.values())
conf = allm.groupby("_sha")["_y"].nunique().gt(1)
record("mooney:identical_image_conflicting_labels", "PASS" if not conf.any() else "FAIL",
       f"{int(conf.sum())} identical images carrying both labels (any split)")
H = {k: set(mo[k]["_sha"]) for k in mo}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    ov = H[a] & H[b]
    record(f"mooney:sha256_identity_{a}_{b}", "PASS" if not ov else "FAIL", f"{len(ov)} identical images across splits")
for k, df in mo.items():
    df[["_path", "_fname", "_pid", "_y", "_sha", "final_split"]].rename(
        columns={"_path": "filepath", "_fname": "filename", "_pid": "patient_id_heuristic", "_y": "y", "_sha": "sha256"}
    ).to_csv(OUT_DIR / f"mooney_{k}_audited.csv", index=False)

### 1e. Near-duplicate candidates (dHash) — WARN, inspect, never auto-delete

In [ ]:
def dhash(path, size=8):
    a = np.asarray(Image.open(path).convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    return np.packbits((a[:, 1:] > a[:, :-1]).flatten()).view(">u8")[0]
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
if RUN_NEAR_DUP:
    for df in mo.values(): df["_dh"] = [dhash(p) for p in df["_abs"]]
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        A, B = mo[a]["_dh"].to_numpy(np.uint64), mo[b]["_dh"].to_numpy(np.uint64)
        pairs = []
        for i0 in range(0, len(A), 512):
            x = np.bitwise_xor(A[i0:i0+512, None], B[None, :]).astype(">u8").view(np.uint8).reshape(-1, len(B), 8)
            d = POP[x].sum(-1); ii, jj = np.where(d <= DHASH_MAX_HAMMING)
            pairs += [(mo[a]._path.iat[i0+i], mo[b]._path.iat[j], int(d[i, j])) for i, j in zip(ii, jj)]
        pd.DataFrame(pairs, columns=[f"{a}_path", f"{b}_path", "hamming"]).to_csv(OUT_DIR / f"near_dup_candidates_{a}_{b}.csv", index=False)
        record(f"mooney:near_dup_{a}_{b}", "PASS" if not pairs else "WARN",
               f"{len(pairs)} candidate pairs — inspect visually, no automatic removal", blocking=False)

## 2. NIH — lock cohorts from metadata rules (no model output exists or is read)

In [ ]:
meta = pd.read_csv(NIH_ROOT / "Data_Entry_2017.csv")
meta["labels"] = meta["Finding Labels"].astype(str).str.split("|").apply(set)
meta["age"] = pd.to_numeric(meta["Patient Age"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
nih_files = {p.name: p for p in NIH_ROOT.glob("images_*/images/*.png")}
record("nih:metadata", "INFO", f"{len(meta)} metadata rows, {len(nih_files)} image files on disk", blocking=False)

def age_reason(a):
    if pd.isna(a): return "missing_or_unparseable"
    if a < AGE_MIN: return "age_lt_18"
    if a > AGE_MAX: return "age_gt_100"
    return "included"
meta["age_status"] = meta["age"].apply(age_reason)

is_pos  = meta["labels"].apply(lambda L: "Pneumonia" in L)
is_nf   = meta["labels"].apply(lambda L: L == {"No Finding"})
is_hard = meta["labels"].apply(lambda L: "Pneumonia" not in L and bool(L & HARD_NEG_FINDINGS))

DEFS = {  # name: (view, negative mask, negative label string)
    "OOD_AP":      ("AP", is_nf,   "NO_FINDING"),
    "OOD_PA":      ("PA", is_nf,   "NO_FINDING"),
    "OOD_HARD_AP": ("AP", is_hard, "NON_PNEUMONIA_ABNORMAL"),
}
FINGERPRINTS, EXCL, COHORTS = {}, [], {}
for base, (view, negmask, negstr) in DEFS.items():
    v = meta["View Position"] == view
    elig = meta[v & (is_pos | negmask)].copy()
    elig["y"] = is_pos[elig.index].astype(int)
    elig["label"] = np.where(elig["y"] == 1, "PNEUMONIA", negstr)
    for suffix, role, frame in [("ALLAGE", "sensitivity", elig), ("ADULT", "primary", elig[elig.age_status == "included"])]:
        name = f"{base}_{suffix}"
        miss = [f for f in frame["Image Index"] if f not in nih_files]
        out = pd.DataFrame({
            "dataset": "NIH_ChestXray14", "cohort": name, "analysis_role": role,
            "filename": frame["Image Index"].values,
            "filepath": [str(nih_files.get(f, "")) for f in frame["Image Index"]],
            "patient_id": frame["Patient ID"].values, "follow_up": frame["Follow-up #"].values,
            "age": frame["age"].values, "sex": frame["Patient Gender"].values, "view": frame["View Position"].values,
            "finding_labels": frame["Finding Labels"].values, "y": frame["y"].values, "label": frame["label"].values,
        }).sort_values("filename").reset_index(drop=True)
        out.to_csv(COHORT_DIR / f"{name}.csv", index=False)
        COHORTS[name] = out
        FINGERPRINTS[name] = hashlib.sha256("\n".join(out.filename).encode()).hexdigest()
        record(f"{name}:files_exist", "PASS" if not miss else "FAIL", f"{len(miss)} images not on disk")
        record(f"{name}:view", "PASS" if (out.view == view).all() else "FAIL", f"all {view}")
        both = int(out.groupby("patient_id")["y"].nunique().gt(1).sum())
        record(f"{name}:characteristics", "INFO",
               f"[{role}] images={len(out)} (pos={int(out.y.sum())}, neg={int((1-out.y).sum())}), "
               f"patients={out.patient_id.nunique()}, patients_in_both_classes={both}, "
               f"age median={out.age.median():.0f} [IQR {out.age.quantile(.25):.0f}–{out.age.quantile(.75):.0f}], "
               f"female={(out.sex == 'F').mean():.1%}", blocking=False)
        record(f"{name}:patients_in_both_classes", "PASS" if both == 0 else "WARN",
               f"{both} patients kept (external-only); patient-level bootstrap required", blocking=False)
    # exclusion accounting (ALLAGE → ADULT)
    adult_pids = set(elig.loc[elig.age_status == "included", "Patient ID"])
    for reason in ["age_lt_18", "age_gt_100", "missing_or_unparseable"]:
        ex = elig[elig.age_status == reason]
        pids = set(ex["Patient ID"])
        EXCL.append(dict(cohort=f"{base}_ADULT", reason=reason,
                         images_excluded=len(ex), images_excluded_pos=int(ex.y.sum()), images_excluded_neg=int((1-ex.y).sum()),
                         patients_with_excluded_images=len(pids),
                         patients_fully_excluded=len(pids - adult_pids)))
    EXCL.append(dict(cohort=f"{base}_ADULT", reason="TOTAL_EXCLUDED",
                     images_excluded=int((elig.age_status != "included").sum()),
                     images_excluded_pos=int(elig.loc[elig.age_status != "included", "y"].sum()),
                     images_excluded_neg=int((1 - elig.loc[elig.age_status != "included", "y"]).sum()),
                     patients_with_excluded_images=elig.loc[elig.age_status != "included", "Patient ID"].nunique(),
                     patients_fully_excluded=len(set(elig.loc[elig.age_status != "included", "Patient ID"]) - adult_pids)))
excl = pd.DataFrame(EXCL); excl.to_csv(OUT_DIR / "cohort_exclusion_report.csv", index=False)
print(excl.to_string(index=False))
for name, df in COHORTS.items():
    ok = not df.empty and df.y.nunique() == 2
    record(f"{name}:both_classes_present", "PASS" if ok else "FAIL", f"n={len(df)}")
for base in DEFS:
    adult = COHORTS[f"{base}_ADULT"]
    ok = adult.age.between(AGE_MIN, AGE_MAX).all()
    record(f"{base}_ADULT:age_bounds", "PASS" if ok else "FAIL", f"all ages within [{AGE_MIN}, {AGE_MAX}]")

In [ ]:
for name, fpv in FINGERPRINTS.items():
    exp = EXPECTED_FINGERPRINTS.get(name)
    record(f"{name}:fingerprint_matches_offline_lock", "PASS" if fpv == exp else "FAIL",
           "identical image list to the cohort locked on 2026-10-05" if fpv == exp else
           "image list differs from the offline lock — do not train; send audit_checks.csv for review")

### 2a. Cross-check Notebook 01 NIH manifests (if present) — superseded by the locked cohorts above

In [ ]:
n01 = [p for p in NIH_N01_MANIFEST_DIR.glob("nih_ood_*.csv") if "summary" not in p.name.lower()]
record("nih:n01_manifests_found", "INFO", f"{[p.name for p in n01]}", blocking=False)
allage_pos = {n: set(COHORTS[f"{n}_ALLAGE"].query("y==1").filename) for n in DEFS}
allage_neg = {n: set(COHORTS[f"{n}_ALLAGE"].query("y==0").filename) for n in DEFS}
for p in n01:
    df = pd.read_csv(p)
    try:
        fc = find_col(df, FNAME_COLS, required=False)
        names = (df[fc] if fc else df[find_col(df, PATH_COLS)]).astype(str).apply(lambda s: Path(s).name)
        y = labels_of(df, f"n01:{p.stem}")
    except Exception as e:
        record(f"n01:{p.stem}:readable", "WARN", f"could not parse: {e}", blocking=False); continue
    best = max(DEFS, key=lambda n: len(set(names) & (allage_pos[n] | allage_neg[n])))
    viol = int(((y == 1) & ~names.isin(allage_pos[best])).sum() + ((y == 0) & ~names.isin(allage_neg[best])).sum())
    record(f"n01:{p.stem}:consistent_with_{best}", "PASS" if viol == 0 else "WARN",
           f"{viol} rows violate the {best} rule. Downstream notebooks must use locked_cohorts/, not this file.",
           blocking=False)

### 2b. NIH ↔ Mooney SHA-256 disjointness

In [ ]:
if HASH_NIH_IMAGES:
    names = sorted(set().union(*[set(d.filename) for d in COHORTS.values()]) & set(nih_files))
    nih_hash = {n: sha256(nih_files[n]) for n in names}
    mh = set().union(*H.values())
    ov = [n for n, h in nih_hash.items() if h in mh]
    record("nih:sha256_disjoint_from_mooney", "PASS" if not ov else "FAIL", f"{len(ov)} NIH images identical to Mooney images")
    pd.Series(nih_hash, name="sha256").rename_axis("filename").to_csv(OUT_DIR / "nih_cohort_sha256.csv")
else:
    record("nih:sha256_disjoint_from_mooney", "WARN", "skipped (HASH_NIH_IMAGES=False)", blocking=False)

### 2c. Visual inspection sheet — closest cross-split near-duplicate candidates (inspection only, nothing is removed)

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
cand = []
for a, b in [("train", "test"), ("val", "test"), ("train", "val")]:
    f = OUT_DIR / f"near_dup_candidates_{a}_{b}.csv"
    if f.exists():
        d = pd.read_csv(f)
        if len(d):
            d.columns = ["path_a", "path_b", "hamming"]; d["pair"] = f"{a}-{b}"; cand.append(d)
cand = pd.concat(cand).sort_values("hamming").head(24) if cand else pd.DataFrame()
if len(cand):
    fig, ax = plt.subplots(len(cand), 2, figsize=(6, 3 * len(cand)))
    ax = np.atleast_2d(ax)
    for r, (_, row) in enumerate(cand.iterrows()):
        for c, pth in enumerate([row.path_a, row.path_b]):
            pth = Path(pth) if Path(pth).is_absolute() else MOONEY_ROOT / pth
            ax[r, c].imshow(Image.open(pth).convert("L"), cmap="gray"); ax[r, c].axis("off")
            ax[r, c].set_title(f"{row.pair} d={row.hamming}\n{pth.name}", fontsize=7)
    plt.tight_layout(); plt.savefig(OUT_DIR / "near_dup_inspection_sheet.png", dpi=110); plt.close()
    record("mooney:near_dup_inspection_sheet", "INFO", f"{len(cand)} closest pairs rendered for visual review", blocking=False)

## 3. Verdict, report, ZIP

In [ ]:
VERDICT = finalize()